# S4 Square Particle

This notebook constructs and solves one periodic square particle structure using S4.

This calculation is divided into:

1. The physical model which includes its dimensions, materials and source.
2. The S4 numerical configuration.
3. A function that constructs S4 simulation.
4. A function that calculates the reflectance and transmittance.
5. Results for s- and p- polarised incidence


In [1]:
import S4

from dataclasses import dataclass
from pathlib import Path

import yaml

from dataclasses import replace
from pathlib import Path

import csv

In [ ]:
@dataclass
class SquareParticleModel:
    """Physical description of one periodic square particle structure.

    All lengths are expressed in nanometers. Angles are expressed in degrees because
    that is what S4 expects. 

    Attributes
    ----------
    period_nm: 
        Period of the square lattice in both x and y
    particle_side_nm:
        Full side length of the square particle.
    planarization_thickness_nm:
        Thickness of the homogeneous planarization layer.
    patterned_layer_thickness_nm:
        Thickness of the layer containing the square particle. 
    wavelength_nm:
        Free-space wavelength.
    theta_deg:
        Polar incidence angle measured from the surface normal.
    phi_deg:
        Azimuthal incidence angle.
    incidence_epsilon:
        Permittivity of the semi-infinite incidence material
    planarization_epsilon:
        Permittivity of the homogeneous planarization layer
    background_epsilon:
        Permittivity of the material surrounding the square particle
    particle_epsilon:
        Permittivity of the square particle.
    transsmission_epsilon:
        Permittivity of the semi-infinite transmission material
    square_corner_radius: 
        Corner rounding parameter used by MetaRCWA. The S4 backened 
        doesn't currently use this value but it remains in the shared
        YAML model definition so the same configuration can be loaded
        by all solver backends.
    """

    period_nm: float
    particle_side_nm: float
    planarization_thickness_nm: float
    patterned_layer_thickness_nm: float
    wavelength_nm: float
    theta_deg: float
    phi_deg: float
    incidence_epsilon: complex
    planarization_epsilon: complex
    background_epsilon: complex
    particle_epsilon: complex
    transmission_epsilon: complex
    square_corner_radius: float

    @classmethod
    def from_dict(cls, data:dict) -> "SquareParticleModel":
        """Create a physical model from YAML-derived dictionary data."""

        # Copy the dictionary so we do not modify the original data
        data = dict(data)

        epsilon_list = (
            "incidence_epsilon",
            "planarization_epsilon",
            "background_epsilon", 
            "particle_epsilon",
            "transmission_epsilon",
        )

        # Convert each {real:..., image:...} dictionary into 
        # an ordinary Python complex number
        for epsilon in epsilon_list:
            components = data[epsilon]

            data[epsilon] = complex(
                components["real"],
                components.get("imag",0.0)
            )

        return cls(**data)

In [3]:
@dataclass
class S4Config:
    """Numerical settings used directly by S4

    Attributes
    ----------
    num_basis:
        Maximum number of Fourier orders to retain. Due to truncation, 
        the actual number may be different.
    lattice_truncation:
        While num_basis gives how many fourier orders to approximately retain,
        lattice truncation says how to choose them. 
        In S4, this can be 'Circular' or 'Parallelogramic'.
    """

    num_basis: int
    lattice_truncation: str
    DiscretizedEpsilon: bool
    DiscretizationResolution: int

    def __post_init__(self) -> None:
        """Check that the supplied settings are valid."""

        if self.num_basis <=0:
            raise ValueError("num_basis must be positive")

        allowed_truncations = {
            "Circular",
            "Parallelogramic"
        }

        if self.lattice_truncation not in allowed_truncations:
            raise ValueError(
                "lattice_truncation must be "
                "'Circular' or 'Parallelogramic'"
            )

    @classmethod
    def from_dict(cls, data:dict) -> "S4Config":
        "Create the numerical settings from YAML derived dictionary data."

        # Copy the dictionary so you don't modify the original data
        data = dict(data)

        return cls(**data)

In [4]:
config_path = Path("../src/configs/square_particle.yaml")

with config_path.open("r") as file:
    yaml_data = yaml.safe_load(file)

In [5]:
model = SquareParticleModel.from_dict(
    yaml_data["model"]
)

config = S4Config.from_dict(
    yaml_data["s4"]
)

## S4 Numerical Configuration

The physical model describes what is being simulated. The S4 configuration describes how S4 approximates the solution.

`num_basis` is the requested maximum number of in-plane Fourier orders.
`lattice_truncation` controls how those orders are selected in reciprocal space.

Other S4 formulation options remain at their documented defaults for this initial baseline.

## Constructing the S4 simulation

S4 requires the following construction sequence:

1. Define the periodic lattice and Fourier basis
2. Define the materials
3. Add the layers in physical stack order
4. Add the square region to the patterned layer
5. Set the frequency

The semi-infinite incidence and transmission spaces are represented as layers with zero thickness.

In [6]:
def build_s4_simulation(
        model: SquareParticleModel,
        config: S4Config
):
    """Construct one S4 simulation from a physical model and configuration.

    Paramters
    ---------
    model:
        Physical square-particle structure and source.
    config:
        Numerical settings used by S4.
    
    Returns
    -------
    simulation:
        A fully constructed S4 simulation. The incident polarization has
        not yet been selected.
    """

    # The real-space lattice vectors define a square periodic unit cell
    lattice = (
        (model.period_nm, 0.0),
        (0.0, model.period_nm)
    )

    # Create an empty S4 simulation with the requested Fourier basis
    S = S4.New(
        Lattice=lattice,
        NumBasis=config.num_basis)

    # Choose how S4 chooses reciprocal-lattice/ Fourier orders
    S.SetOptions(LatticeTruncation = config.lattice_truncation,
                DiscretizedEpsilon=config.DiscretizedEpsilon,
                DiscretizationResolution=config.DiscretizationResolution)

    # Define the materials used by the stack
    S.SetMaterial(Name='Incidence_material',
                  Epsilon = model.incidence_epsilon)
    S.SetMaterial(Name='Planarization_material',
                  Epsilon = model.planarization_epsilon)
    S.SetMaterial(Name='Particle_material',
                  Epsilon = model.particle_epsilon)
    S.SetMaterial(Name='Background_material',
                  Epsilon=model.background_epsilon)
    S.SetMaterial(Name='Transmission_material',
                  Epsilon=model.transmission_epsilon)

    # Add the semi-infinite incidence medium
    S.AddLayer(Name='Incidence',
               Thickness=0.0,
               Material='Incidence_material')

    # Add the homogeneous planarization layer
    S.AddLayer(Name='Planarization',
               Thickness=model.planarization_thickness_nm,
               Material='Planarization_material')

    # Initially fill the patterned layer with the background material
    S.AddLayer(Name='Patterned_layer',
               Thickness = model.patterned_layer_thickness_nm,
               Material='Background_material')

    # Add the semi-infinite transmission medium
    S.AddLayer(
        Name='Transmission',
        Thickness=0.0,
        Material='Transmission_material'
    )

    # MetaShapes previously used full widths and height.
    # S4 requires distances from the centre to the rectangle edges
    particle_halfwidth = model.particle_side_nm / 2.0

    # S4's unit-cell origin is at its centre so a centred square uses
    # centre=(0,0)
    S.SetRegionRectangle(
        Layer='Patterned_layer',
        Material='Particle_material',
        Center = (0.0, 0.0),
        Angle=0.0,
        Halfwidths=(
            particle_halfwidth,
            particle_halfwidth
        )
    )

    # S4 uses ordinary frequency. When all lengths use nanometers,
    # frequency is specified in inverse nanometers.
    S.SetFrequency(
        1.0 / model.wavelength_nm
    )

    return S

In [7]:
S = build_s4_simulation(
    model=model,
    config=config
)

## Reflection and Transmission

S4's `GetPowerFlux()` returns the integrated forward and backward Poynting flux through one unit cell surface.

For the incidence layer:

- forward flux is the incidence power
- backward flux is the reflected power travelling in the negative z direction

For the transmission layer:

- forward flux is the transmitted power

In [8]:
def reflectance_and_transmittance(
        simulation,
        polarization: str,
        theta_deg: float,
        phi_deg: float,
) -> dict[str,complex | float]:
    """Calculate total reflectance and transmittance.

    Parameters
    ----------
    simulation:
        A fully constructed S4 simulation.
    polarization:
        Incident polarization, either s or p.
    theta_deg:
        Polar incidence angle in degrees, measured from the normal.
    phi_deg: 
        Azimuthal incidence angle in degrees.
    
    Returns
    -------

    A dictionary with:
    - r0, reflection coefficient
    - t0, transmission coefficient
    - R, total reflectance
    - T, total transmittance
    - R_zero: Reflectance for zero order
    - T_zero: Reflectance for zero order 
    """

    if polarization == 's':
        s_amplitude = 1.0 + 0.0j
        p_amplitude = 0.0 + 0.0j
    if polarization == 'p':
        s_amplitude = 0.0 + 0.0j
        p_amplitude = 1.0 + 0.0j

    # Solve one simulation and extract complex reflection coefficient for
    # 0th order reflected wave

    # S4 GetBasisSet() returns the pairs of reciprocal
    # lattice coordinates used in that basis so they 
    # are integers. Will use this get desired index of 
    # all the amplitudes and etc being calculated

    # Returns a tuple containing tuples of 2
    # (0,0),....
    orders = simulation.GetBasisSet()

    # Python tuples have .index() method to search for
    # a value and return its position
    zero_order_index = orders.index((0,0))
    number_of_orders = len(orders)

    # Illuminate the structure using the zeroth incidence diffraction order
    simulation.SetExcitationPlanewave(
        IncidenceAngles=(
            theta_deg,
            phi_deg
        ),
        sAmplitude=s_amplitude,
        pAmplitude=p_amplitude,
        Order=zero_order_index
    )

    # The zero_order_index for p polarisation is different to s
    if polarization == 's':
        amplitude_index = zero_order_index
    else:
        amplitude_index = (
            number_of_orders + zero_order_index
        )

    # Give the coefficients of the modes travelling forwards and backwards
    # in the incidence medium
    incidence_forwards, incidence_backwards = simulation.GetAmplitudes(
        Layer = 'Incidence',
        zOffset = 0
    )

    transmitted_forward,_= simulation.GetAmplitudes(
        Layer = 'Transmission',
        zOffset = 0
    )
    # We want the first diffraction order (0,0)
    incidence_amplitude = incidence_forwards[amplitude_index]
    reflected_amplitude = incidence_backwards[amplitude_index]
    transmitted_amplitude = transmitted_forward[amplitude_index]

    # Calculate the refelection coefficient
    r0 = reflected_amplitude / incidence_amplitude
    t0 = transmitted_amplitude / incidence_amplitude

    # Although reflection coefficient can be squared to get reflectance,
    # Cannot apply the amplitude method to transmission as the amplitude of 
    # the incident and transmitted modes below to different materials and 
    # therefore their power normalisation factors which depend on things like
    # material won't cancel unlike for reflectance. 

    # Use GetPowerFlux to get overall transmission and reflectance

    incident_power, reflected_power = simulation.GetPowerFlux(
        Layer = 'Incidence',
        zOffset = 0
    )

    transmitted_power,_=simulation.GetPowerFlux(
        Layer='Transmission',
        zOffset=0
    )

    R = -reflected_power.real / incident_power.real
    T = transmitted_power.real / incident_power.real

    # Each item in GetPowerFluxByOrder for a specific layer is: 
    # (forward power flux, backward power flux) for each diffraction order
    incident_flux_by_order = simulation.GetPowerFluxByOrder(
        Layer = 'Incidence',
        zOffset = 0
    )

    transmission_flux_by_order=simulation.GetPowerFluxByOrder(
        Layer = 'Transmission',
        zOffset = 0
    )

    # Reflection is the backward flux in the incidence layer
    # Physical time-averaged EM power is obtained from the real part of the
    # complex Poynting vector
    _,reflected_zero_flux = incident_flux_by_order[zero_order_index]

    # Transmission is the forward flux in the transmission layer
    transmitted_zero_flux,_ = transmission_flux_by_order[zero_order_index]

    R_zero = (
        -reflected_zero_flux.real / incident_power.real
    )

    T_zero = (
        transmitted_zero_flux.real / incident_power.real
    )
    return {
        'r0': r0,
        't0': t0,
        'R': R,
        'T': T,
        'R_zero': R_zero,
        'T_zero': T_zero
    }

In [9]:
s_result = reflectance_and_transmittance(
    simulation=S,
    polarization='s',
    theta_deg = model.theta_deg,
    phi_deg=model.phi_deg
)

p_result = reflectance_and_transmittance(
    simulation=S,
    polarization='p',
    theta_deg=model.theta_deg,
    phi_deg=model.phi_deg
)

Rs = s_result['R']
Ts = s_result['T']
Rp = p_result['R']
Tp = p_result['T']

print(f"Rs = {Rs: .10f}")
print(f"Rp = {Rp: .10f}")
print(f"Ts = {Ts: .10f}")
print(f"Tp = {Tp: .10f}")

print("S4 terms:", len(S.GetBasisSet()))

Rs =  0.0203826390
Rp =  0.0203826390
Ts =  0.9796173610
Tp =  0.9796173610
S4 terms: 25


In [10]:
# range() excludes the final value, so use 101 to include 100
requested_orders = list(
    range(5,116,5)
)

print(requested_orders)

[5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100, 105, 110, 115]


In [12]:
s4_Rs_values = []
s4_actual_orders = []

for requested_num_basis in requested_orders:

    # Copy the original S4 configuration, changing only num_basis
    sweep_config = replace(
        config,
        num_basis = requested_num_basis
    )

    # Rebuild simulation because Fourier basis has changed
    sweep_simulation = build_s4_simulation(
        model=model,
        config=sweep_config
    )

    # Calculate reflection for s-polarised incidence
    sweep_result = reflectance_and_transmittance(
        simulation=sweep_simulation,
        polarization = "s",
        theta_deg=model.theta_deg,
        phi_deg=model.phi_deg
    )

    Rs = float(sweep_result["R"])

    # S4 may retain a slightly different number from the 
    # request
    actual_num_basis = len(
        sweep_simulation.GetBasisSet()
    )

    # Store this simulation's results
    s4_Rs_values.append(Rs)
    s4_actual_orders.append(actual_num_basis)

    print(
        f"Requested: {requested_num_basis:3d}, "
        f"actual: {actual_num_basis:3d}, "
        f"Rs: {Rs:.10f}"
    )

Requested:   5, actual:   5, Rs: 0.0209910113
Requested:  10, actual:   9, Rs: 0.0206575476
Requested:  15, actual:  13, Rs: 0.0205340740
Requested:  20, actual:  13, Rs: 0.0205340740
Requested:  25, actual:  25, Rs: 0.0203826390
Requested:  30, actual:  29, Rs: 0.0203794491
Requested:  35, actual:  29, Rs: 0.0203794491
Requested:  40, actual:  37, Rs: 0.0203736133
Requested:  45, actual:  45, Rs: 0.0203698659
Requested:  50, actual:  49, Rs: 0.0203198213
Requested:  55, actual:  49, Rs: 0.0203198213
Requested:  60, actual:  57, Rs: 0.0202606287
Requested:  65, actual:  61, Rs: 0.0202602934
Requested:  70, actual:  69, Rs: 0.0202471282
Requested:  75, actual:  69, Rs: 0.0202471282
Requested:  80, actual:  69, Rs: 0.0202471282
Requested:  85, actual:  81, Rs: 0.0202288619
Requested:  90, actual:  89, Rs: 0.0202087068
Requested:  95, actual:  89, Rs: 0.0202087068
Requested: 100, actual:  97, Rs: 0.0202058510
Requested: 105, actual: 101, Rs: 0.0202038957
Requested: 110, actual: 109, Rs: 0

In [13]:
results_directory = Path("plots/csv")

results_directory.mkdir(
    parents=True,
    exist_ok=True
)

s4_output_path = (
    results_directory
    / "s4_fourier_convergence.csv"
)

with s4_output_path.open(
    "w",
    newline="", 
) as file:

    writer = csv.writer(file)

    writer.writerow([
        "requested_orders",
        "actual_orders",
        "Rs"
    ])

    writer.writerows(
        zip(
            requested_orders,
            s4_actual_orders,
            s4_Rs_values,
            strict=True
        )
    )

print(f"Saved results to {s4_output_path}")

Saved results to plots/csv/s4_fourier_convergence.csv
